In [1]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import re
import openpyxl
# !pip install PyPDF2
from PyPDF2 import PdfReader                    
import warnings
import difflib
import datetime
import re
warnings.filterwarnings("ignore", category=UserWarning, module="camelot.parsers.base")



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'IN RBI' ## change to current controller name


print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running IN RBI Web Scraping Tool v.1.1


In [3]:

# Start Chrome driver with download folder set to tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {
    "plugins.always_open_pdf_externally": True,
    "download.prompt_for_download": False,
    "download.default_directory": tempfolder
}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

#Creating dictionarywith ingegcodes and their respective URLs
regdict = {
            # 'IN RBI 1': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#NB', 'Nationalised Banks'),
            # 'IN RBI 2': ('https://www.rbi.org.in/scripts/bs_viewcontent.aspx?Id=2463', 'List of Banks permitted to provide Mobile Banking Service in India'),
            # 'IN RBI 3': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#SBIA', 'SBI'),
            # 'IN RBI 4': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#IB', 'Indian Banks'),
            # 'IN RBI 5': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#FB', 'Foreign Banks'),
            # 'IN RBI 6': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#SCB', 'State Co-operative Banks'),
            # 'IN RBI 7': ('https://rbidocs.rbi.org.in/rdocs/Content/pdfs/schedulecoop.pdf', 'Scheduled Urban Cooperative Banks'),
			# 'IN RBI 8': ('https://rbidocs.rbi.org.in/rdocs/Content/pdfs/nonschedulecoop.pdf', 'Non-Scheduled Urban Co-operative Banks'),
			# 'IN RBI 9': ('https://rbidocs.rbi.org.in/rdocs/Content/pdfs/DCCB20141702.pdf', 'District Central Cooperative Bank'),
			'IN RBI 10': ('https://www.rbi.org.in/commonman/English/Scripts/BanksInIndia.aspx#rrb', 'Regional Rural Banks'),
			# 'IN RBI 11': ('https://www.rbi.org.in/commonman/English/Scripts/Content.aspx?id=2139', 'List of agency banks'),
			# 'IN RBI 12': ('https://rbi.org.in/Scripts/PublicationsView.aspx?id=12043', 'List of Payment System Operators'),
			# 'IN RBI 13': ('https://rbi.org.in/Scripts/bs_viewcontent.aspx?Id=2491', 'List of banks permitted to issue pre-paid cards in India'),
			# 'IN RBI 14': ('https://www.rbi.org.in/scripts/bs_viewcontent.aspx?Id=3657', 'List of Scheduled Commercial Bank'),
			# 'IN RBI 15': ('https://www.rbi.org.in/commonman/English/Scripts/Content.aspx?id=336', 'Banks Authorised to Import Gold/Silver'),
			# 'IN RBI 16': ('https://rbi.org.in/Scripts/BS_NBFCList.aspx', 'Non Banking Financial Companies')
		   }

#Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame
sqldict={    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],     'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
'InternalID_2_type': [],     'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],     'Address_1': [], 'Address_2': [], 'City': [], 
'Zip': [], 'Cntry': [], 'Phone': [],     'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [],     'RegulationDate': [], 'CancellationDate': [], 
'RegCtry': [], 'RegCode' : [],     'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [],     'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],     'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')


In [4]:
def add_record(record):
    for key in sqldict:
        sqldict[key].append(record.get(key, ''))

def append_record(bank_name, bank_address, pincode="", website="", phone=""):
    record = {
        'Name': bank_name,
        'Address_1': bank_address,
        'ListProcessDate': processdate,
        'RegCtry': 'IN',
        'RegCode': 'RBI',
        'ListCode': reg.split(' ')[-1],
        'ListName': list_name,
        'RegulationType': "Regulated",
        'Zip': pincode,
        'Website': website,  
        'Phone': phone       
    }
    add_record(record)

def process_bank_details(td):
    """Extracts bank name and address from a given <td> element."""
    for br in td.find_all("br"):
        br.replace_with("\n")
    text = td.get_text("\n", strip=True)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if not lines:
        return None, None
    bank_name = re.sub(r'\s+', ' ', lines[0].split(",")[0].strip())
    bank_address = re.sub(r'\s+', ' ', " ".join(lines[1:]).replace(",", ""))
    bank_address = re.sub(r'\.+$', '', re.sub(r'\.+$', '', bank_address)).strip()

    return bank_name, bank_address

def normalize_text(s):
    return s.lower().strip()


def convert_date(date_str):
    """
    Converts the date string into American format
    If the date string contains "revocation order issued"
    
    It handles formats like:
      - "17.05.2017 Note: The entity was granted approval to commence operations w.e.f 01.12.2016"
      - "May 01, 2011"
      - "November 01, 2013"
      
    In a mixed string, the function uses the part before any "note:" and ignores any
    parenthetical content
    """
    s = date_str.strip()
   

    # Remove any extra note text after "Note:" and remove parentheses 
    s = s.split("Note:")[0].strip()
    s = re.sub(r'\s*\(.*?\)', '', date_info).strip()

    for fmt in ("%d.%m.%Y", "%B %d, %Y", "%b %d, %Y"):
        try:
            dt = datetime.datetime.strptime(s, fmt)
            return dt.strftime("%m/%d/%Y")
        except Exception:
            continue
    # If parsing fails, return the cleaned string
    return s

def clean_pdf_address(address):
    """
    Splits the address on commas, removes duplicate (case‑insensitive) tokens,
    and rejoins them. Adjust the splitting if the PDF extraction uses other delimiters.
    """
    tokens = [token.strip() for token in address.split(',') if token.strip()]
    seen = []
    for token in tokens:
        if token.lower() not in [s.lower() for s in seen]:
            seen.append(token)
    return ", ".join(seen)

def merge_text(old_val, new_val, sep=" "):
    """
    Joins two text values with a separator.
    If one of them is empty, returns the other.
    Normalizes whitespace.
    """
    old_val = old_val.strip() if old_val else ""
    new_val = new_val.strip() if new_val else ""
    if old_val and new_val:
        return " ".join([old_val, new_val]).replace("  ", " ").strip()
    else:
        return old_val or new_val
    

# if address has two zip codes, split into two records.
def split_double_zip_record(bank_name_extracted, address_extracted):
    zip_pattern = re.compile(r'\b\d{3}\s?\d{3}\b')    
    zips = list(zip_pattern.finditer(address_extracted))
    if len(zips) >= 1:
        # Use the first zip match to get the split point.
        first_zip_match = zips[0]
        split_index = first_zip_match.end()
        # If a comma (and optionally extra numbers/dots) follows the zip, advance the split
        if address_extracted[split_index:split_index+1] == ',':
            extraneous = re.match(r'\s*,\s*\d{1,2}[\s\.]*', address_extracted[split_index:])
            if extraneous:
                split_index += extraneous.end()
            else:
                split_index += 1
        first_part = address_extracted[:split_index].strip(" ,")
        second_part = address_extracted[split_index:].strip(" ,")
        # Remove any leading punctuation/spaces from the second part
        second_part = re.sub(r'^[\.,\s]+', '', second_part)
        # Extract the bank name from the start of second_part (up to and including "Ltd")
        m_bank = re.search(r'^(.*?\bLtd\.?)', second_part, re.I)
        if m_bank:
            bank_name_new = m_bank.group(1).strip()
            second_addr = second_part[m_bank.end():].strip(" ,")
            rec1 = {'Name': bank_name_extracted, 'Address_1': first_part}
            rec2 = {'Name': bank_name_new, 'Address_1': second_addr}
            return rec1, rec2
    return None

def fix_bangalore_case(bank_name_extracted, address_extracted):
    # If bank_name contains "Bangalore District and Bangalore Rural District" then split at "Co-op"
    if re.search(r'Bangalore District and Bangalore Rural District', bank_name_extracted, re.I):
        m = re.search(r'(.*?)(Co[-\s]*op\.?\s*Central\s*Bank\s*Ltd\.?)', bank_name_extracted, re.I)
        if m:
            bank_name_extracted = m.group(1).strip()
            extra = m.group(2).strip()
            # Prepend the extra fragment to the address
            address_extracted = extra + (", " + address_extracted if address_extracted else "")
    return bank_name_extracted, address_extracted

def remove_leading_punctuation(address):
    return re.sub(r'^[\.,\s]+', '', address)

# New extraction function for List 10 cells.

def extract_rrb_bank_details_list10(td):
    if not td:
        return None, "", ""

    text = td.get_text("\n", strip=True)
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.split("\n") if line.strip()]

    website_tag = td.find("a", class_="link1")
    website = website_tag.get("href", "").strip() if website_tag else ""

    if not lines:
        return None, "", website

    # First line: "28. West Bengal Gramin Bank"
    first_line = lines[0]
    m = re.match(r"^\d+\.\s*(.+)$", first_line)
    if m:
        bank_name = m.group(1).strip()
    else:
        bank_name = first_line.strip()

    # Skip "Head Office" if present
    address_lines = []
    for line in lines[1:]:
        if line.lower() == "head office":
            continue
        if line.lower().startswith("website:"):
            continue
        address_lines.append(line)

    full_address = ", ".join(address_lines)
    full_address = re.sub(r"\s+", " ", full_address).strip(" ,")

    return bank_name, full_address, website


def remove_fuzzy_name(name, address, threshold=0.8):
    name = name.lower()
    address = address.lower()

    address_words = address.split()

    for i in range(len(address_words)):
        for j in range(i + 1, len(address_words) + 1):
            candidate = " ".join(address_words[i:j]).lower()
            ratio = difflib.SequenceMatcher(None, name, candidate).ratio()
            if ratio >= threshold:
                to_remove = " ".join(address_words[i:j])
                return address.replace(to_remove, "").strip()
    return address



In [5]:


# Loop through each regulator URL
for reg, (url, list_name) in regdict.items():
    print('Working with {}'.format(reg))
    driver.get(url)
    sleep(4)
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    if reg == "IN RBI 12":

        titles = ["The Chief Executive Officer", "The Chief Executive Office", "Chief Executive Officer","The Chairman & Managing Director", "The Managing Director and Chief Executive Officer","The Managing Director & Chief Executive Officer", "The Managing Director", "Managing Director & CEO",  "The Director",  "The Chairman", "The President" ]
        address_remove = ["of india", "ltd", "limited", "pte", "inc", "pvt"]
        target_table = soup.find("table", class_="tablebg")
        if target_table:
            rows = target_table.find_all("tr")
            if rows:
                print("Processing List 12 with {} rows".format(len(rows)))
                for row in rows[1:]:
                    cols = row.find_all("td")
                    if len(cols) < 5:
                        continue
                    # Use column 0 for serial
                    serial = cols[0].get_text(" ", strip=True)
                    # Extract bank name from column 1
                    name_cell = cols[1].get_text(" ", strip=True)
                    name_cell = re.sub(r"\s*[\(\{].*?[\)\}\*]", "", name_cell).strip()
                    if "Ltd." in name_cell:
                            name_cell = name_cell.split("Ltd.", 1)[0].strip()
                    if "Inc." in name_cell:
                            name_cell = name_cell.split("Inc.", 1)[0].strip()
                            

                    # If name_cell is empty, try fallback extraction
                    address_cel = cols[2].get_text(" ", strip=True)
                    for phrase in titles:
                        address_cel = re.sub(re.escape(phrase), "", address_cel, flags=re.IGNORECASE)
                    address_cel = remove_fuzzy_name(name_cell, address_cel)
                    for word in address_remove:
                        address_cel = re.sub(re.escape(word), "", address_cel, flags=re.IGNORECASE)
              
                    

                    date_cel = cols[4].get_text(" ", strip=True)
                    if not name_cell:
                        name_cell = " ".join(s for s in cols[1].stripped_strings if s)
                    
                    # Process as new record if serial cell has a leading number
                    if serial and re.match(r'^\s*\d+\.', serial):
                        bank_name = name_cell  # Use bank name from column 1
                        address = re.sub(r'^[\s\.,\-&\–\#]+', '', address_cel)
                        date_info = date_cel                        
                        date_info = convert_date(date_cel)
                        date_info = date_info.split("Note:")[0].strip()

                        if "revocation order issued" in date_info.lower():
                            bank_name, address, date_info = "", "", ""

                        if bank_name:
                            record = {
                                'Name': bank_name,
                                'Address_1': address,  
                                'RegulationDate': date_info, 
                                'ListProcessDate': processdate,
                                'RegCtry': 'IN',
                                'RegCode': 'RBI',
                                'ListCode': reg.split(' ')[-1],
                                'ListName': list_name,
                                'RegulationType': "Regulated",
                                'Website': ""
                            }
                            add_record(record)
        else:
            print("Could not locate table for List 12")
        continue  #
    if reg == "IN RBI 16":
        # Locate the cell containing the target label
        #label_cell = soup.find('td', string=re.compile(r'List of NBFCs and ARCs registered with the RBI', re.I))
        label_cell = soup.find('table').find('td')
        if label_cell:
            # Find the Excel link following that cell.
            xlsx_link_tag = label_cell.find_next('a', href=re.compile(r'\.xlsx$', re.I))
            if xlsx_link_tag:
                from urllib.parse import urljoin
                excel_url = urljoin(url, xlsx_link_tag['href'])
                excel_filename = os.path.join(tempfolder, os.path.basename(excel_url))
                # Download the Excel file if not already present
                if not os.path.exists(excel_filename):
                    import requests
                    print("Downloading Excel from", excel_url)
                    r = requests.get(excel_url)
                    with open(excel_filename, 'wb') as f:
                        f.write(r.content)
                # Process the Excel file
                try:
                    df_excel = pd.read_excel(excel_filename, engine="openpyxl")
                except Exception as e:
                    print("Error reading Excel file:", e)
                    continue
                unique_keys = set()
                for idx, row in df_excel.iterrows():
                    # Skip rows that do not have a valid serial (assuming SR No. is numeric)
                    try:
                        serial_val = str(row.iloc[0]).strip()
                        if not serial_val or not re.match(r'^\d+', serial_val):
                            continue
                    except Exception:
                        continue
                    
                    nbfc_name = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ""
                    city = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ""
                    cin = str(row.iloc[5]).strip() if pd.notna(row.iloc[5]) else ""
                    address = str(row.iloc[7]).strip() if pd.notna(row.iloc[7]) else ""
                    email = str(row.iloc[8]).strip() if pd.notna(row.iloc[8]) else ""
                    if not nbfc_name:
                        continue
                    key = (nbfc_name.lower(), cin.lower())
                    if key in unique_keys:
                        continue
                    unique_keys.add(key)

                    if nbfc_name:
                        record = {
                            'Name': nbfc_name,
                            'Address_1': address,
                            'Email': email,
                            'City': city,
                            'InternalID_1': cin,
                            'InternalID_1_type': "Corporate Identification Number",
                            'ListProcessDate': processdate,
                            'RegCtry': 'IN',
                            'RegCode': 'RBI',
                            'ListCode': reg.split(' ')[-1],
                            'ListName': list_name,
                            'RegulationType': "Regulated",
                            'Website': ""
                        }
                        add_record(record)
                try:
                    os.remove(excel_filename)
                    print("Deleted Excel file", excel_filename)
                except Exception as e:
                    print("Error deleting excel file:", e)
            else:
                print("Excel link not found for NBFCs registered list.")
        else:
            print("Label 'List of NBFCs registered with the RBI' not found.")
        continue  # Skip further processing for this regulator.

    if re.search(r'\.pdf$', url, re.I):
        print("PDF detected - extracting table from PDF file")
        
        pdf_filename = os.path.join(tempfolder, os.path.basename(url))
        if not os.path.exists(pdf_filename):
            import requests
            print("Downloading PDF from", url)
            r = requests.get(url)
            with open(pdf_filename, 'wb') as f:
                f.write(r.content)
        
        try:
            import camelot
        except ImportError:
            driver.quit()
            exit(1)
        
        tables = camelot.read_pdf(pdf_filename, pages='all', flavor='stream')
        if tables.n == 0:
            print("No tables detected in PDF.")
        else:
            print("Processing {} tables from PDF".format(tables.n))
            df_pdf = pd.concat([table.df for table in tables], ignore_index=True)
            
            # list 7 
            if reg == "IN RBI 7":
                print("Using List 7 specific mapping")
                records_list = []
                
                header_found = False
                for idx, row in df_pdf.iterrows():
                    if len(row) < 5:
                        continue
                        
                    # Check if this is a header row
                    first_cell = str(row[0]).strip().lower() if pd.notna(row[0]) else ""
                    if re.search(r'sr\. no\.|bank name', first_cell):
                        header_found = True
                        continue
                        
                    # Skip rows until we find the header
                    if not header_found and idx < 4:
                        continue

                    # Extract data from correct columns
                    bank_name_raw = row[1].strip() if pd.notna(row[1]) else ""
                    city = row[2].strip() if pd.notna(row[2]) else ""
                    address_raw = row[3].strip() if pd.notna(row[3]) else ""
                    zip_code = row[4].strip() if pd.notna(row[4]) else ""
                    
                    if not bank_name_raw:
                        continue
                    
                    bank_address = clean_pdf_address(address_raw)
                    
                    rec = {
                        'Name': bank_name_raw,
                        'Address_1': bank_address,
                        'City': city,
                        'Zip': zip_code,
                        'ListProcessDate': processdate,
                        'RegCtry': 'IN',
                        'RegCode': 'RBI',
                        'ListCode': reg.split(' ')[-1],
                        'ListName': list_name,
                        'RegulationType': "Regulated",
                        'Website': ""
                    }
                    records_list.append(rec)
                
                # Remove duplicates keeping most complete record
                final_records = {}
                fields_to_check = ["Name", "Address_1", "City", "Zip"]
                for rec in records_list:
                    key = (normalize_text(rec["Name"]), normalize_text(rec["City"]))
                    completeness = sum(bool(rec[field]) for field in fields_to_check)
                    if key not in final_records or completeness > final_records[key].get("completeness", 0):
                        rec["completeness"] = completeness
                        final_records[key] = rec
                
                # Add deduplicated records
                for rec in final_records.values():
                    rec.pop("completeness", None)
                    add_record(rec)
                    
            elif reg == "IN RBI 8":
                print(f"Using List {reg.split()[-1]} specific mapping")
                cleaned_rows = []
                current_row = None
                header_found = False
                records_list = []
                thiru_pattern = re.compile(r'THIRUVANANTHAPURAM\s+(No\.?[-\s]*[A-Z0-9.-]+.*)', re.IGNORECASE)
                zip_pattern = re.compile(r'\b(\d{6})\b')

                for idx, row in df_pdf.iterrows():
                    # Check for header row and skip it.
                    first_cell = str(row[0]).strip() if pd.notna(row[0]) else ""
                    if re.search(r'sr.*no|bank.*name', first_cell, re.I):
                        header_found = True
                        continue

                    # Skip rows until header is found.
                    if not header_found:
                        continue

                    # Extract values.
                    serial    = str(row[0]).strip() if pd.notna(row[0]) else ""
                    bank_name = str(row[1]).strip() if pd.notna(row[1]) else ""
                    city      = str(row[2]).strip() if pd.notna(row[2]) else ""
                    address   = str(row[3]).strip() if pd.notna(row[3]) else ""
                    zip_code  = str(row[4]).strip() if pd.notna(row[4]) else ""
                    
                    # If the row has a serial, this is a new record
                    if serial:
                        if current_row is not None:
                            # Clean up the address before appending
                            current_row[3] = re.sub(r'\s+', ' ', current_row[3]).strip()
                            cleaned_rows.append(current_row)
                        current_row = list(row)  
                    else:
                        # Continuation row—merge values into the previous record
                        if current_row is not None:
                            # Merge bank name
                            if bank_name:
                                current_row[1] = merge_text(current_row[1], bank_name, sep=" ")
                            # Merge city: if current row’s city is empty or missing the continuation detail
                            if city:
                                current_city = current_row[2].strip() if current_row[2] else ""
                                if current_city:
                                    if city.lower() not in current_city.lower():
                                        current_row[2] = merge_text(current_city, city, sep=", ")
                                else:
                                    current_row[2] = city
                            # Merge address.
                            if address:
                                prev_addr = str(current_row[3]) if pd.notna(current_row[3]) else ""
                                current_row[3] = merge_text(prev_addr, address, sep=", ")
                            # Merge zip code.
                            if zip_code:
                                prev_zip = str(current_row[4]) if pd.notna(current_row[4]) else ""
                                if prev_zip:
                                    if zip_code not in prev_zip:
                                        current_row[4] = merge_text(prev_zip, zip_code, sep=", ")
                                else:
                                    current_row[4] = zip_code

                            # Additionally, if the address field holds a 6-digit zip, extract it.
                            zip_match = zip_pattern.search(current_row[3])
                            if zip_match:
                                extracted_zip = zip_match.group(1)
                                current_row[4] = extracted_zip
                                # Remove the extracted zip from the address field.
                                # current_row[3] = re.sub(r'\b' + re.escape(extracted_zip) + r'\b', current_row[3]).strip()
                    
                            # If the city field contains "THIRUVANANTHAPURAM", process it
                            if "THIRUVANANTHAPURAM" in current_row[2].upper():
                                # Compile pattern with named groups to identify which pattern matched
                                no_pattern = re.compile(r'\b(?P<no>No[.\- ]+|Ltd No[.\- ]+)|(?P<head>Head Office[: ]+)|(?P<num>\d{3}\s+)', re.IGNORECASE)
                                
                                # Find the first match to determine which pattern was found
                                match = no_pattern.search(current_row[2].strip())
                                parts = no_pattern.split(current_row[2].strip(), maxsplit=1)
                                
                                # Set city to just "THIRUVANANTHAPURAM"
                                current_row[2] = "THIRUVANANTHAPURAM"
                                
                                # If there was text after the pattern, add it to address with appropriate prefix
                                if len(parts) > 1:
                                    if match:
                                        if match.group('no'):
                                            prefix = "No."
                                        elif match.group('head'):
                                            prefix = "Head Office: "
                                        elif match.group('num'):
                                            prefix = "No."  
                                        else:
                                            prefix = "No."  # fallback
                                    else:
                                        prefix = "No."  # fallback
                                        
                                    extra_address = prefix + parts[-1].strip()
                                    if current_row[3]:
                                        current_row[3] = merge_text(extra_address, current_row[3], sep=", ")
                                    else:
                                        current_row[3] = extra_address

                # Append the last record if present
                if current_row is not None:
                    current_row[3] = re.sub(r'\s+', ' ', current_row[3]).strip()
                    cleaned_rows.append(current_row)

                # Convert the list of cleaned rows into a DataFrame
                cleaned_df = pd.DataFrame(cleaned_rows, columns=df_pdf.columns)

                for _, row in cleaned_df.iterrows():
                    rec = {
                        'Name': row[1],
                        'City': row[2],
                        'Address_1': row[3],
                        'Zip': row[4],
                        'ListProcessDate': processdate,
                        'RegCtry': 'IN',
                        'RegCode': 'RBI',
                        'ListCode': reg.split(' ')[-1],
                        'ListName': list_name,
                        'RegulationType': "Regulated",
                        'Website': ""
                    }
                    records_list.append(rec)

                # Remove duplicates keeping most complete record
                final_records = {}
                fields_to_check = ["Name", "Address_1", "City", "Zip"]
                for rec in records_list:
                    key = (normalize_text(rec["Name"]), normalize_text(rec["City"]))
                    completeness = sum(bool(rec[field]) for field in fields_to_check)
                    if key not in final_records or completeness > final_records[key].get("completeness", 0):
                        rec["completeness"] = completeness
                        final_records[key] = rec
                
                # Add deduplicated records
                for rec in final_records.values():
                    rec.pop("completeness", None)
                    add_record(rec)
            
            elif reg == "IN RBI 9":
                print("Processing IN RBI 9 PDF for District Central Cooperative Bank records")
                reader = PdfReader(pdf_filename)
                pdf_text = ""
                for page in reader.pages:
                    pdf_text += page.extract_text() + "\n"
                
                # Split text into non-empty lines
                lines = [line.strip() for line in pdf_text.splitlines() if line.strip()]
                content_lines = lines[3:]
                
                records = []
                record_lines = []
                for line in content_lines:
                    if re.match(r'^\d+\.', line):
                        if record_lines:
                            records.append(record_lines)
                        record_lines = [line]
                    else:
                        record_lines.append(line)
                if record_lines:
                    records.append(record_lines)
                
                # Process each record group into a new row
                for rec_lines in records:
                    # Extract the first line (without serial number) and remove dashed text.
                    m = re.match(r'^\d+\.\s*(.*)', rec_lines[0])
                    first_line = m.group(1).strip() if m else rec_lines[0].strip()
                    first_line = re.sub(r'-{5,}', '', first_line).strip()
                    
                    # Initially, set bank name from the first line and address from remaining lines.
                    bank_name_extracted = first_line
                    address_extracted = ", ".join(rec_lines[1:]) if len(rec_lines) > 1 else ""
                    address_extracted = re.sub(r'-{5,}', '', address_extracted).strip()
                    
                    # CASE 1: If the bank name contains extra text after "Ltd", remove it and prepend it to the address
                    pattern_bank = re.compile(r'^(.*?\bLtd\.?)(.*)$', re.I)
                    match_bank = pattern_bank.search(bank_name_extracted)
                    if match_bank:
                        base_name = match_bank.group(1).strip()  # Up to and including "Ltd"
                        extra_from_bank = match_bank.group(2).strip().lstrip(",;:- ")
                        bank_name_extracted = base_name
                        if extra_from_bank:
                            address_extracted = extra_from_bank + (", " if address_extracted else "") + address_extracted
                    
                    # CASE 2: If bank name is empty or does not end with "Ltd", check if the address starts 
                    # with a fragment that ends with "Ltd" and extract it
                    if not bank_name_extracted or not re.search(r'\bLtd\.?$', bank_name_extracted, re.I):
                        pattern_addr = re.compile(r'^(.*?\bLtd\.?)\b[,;:-]?\s*(.*)$', re.I)
                        match_addr = pattern_addr.match(address_extracted)
                        if match_addr:
                            extra_for_name = match_addr.group(1).strip()
                            address_extracted = match_addr.group(2).strip()
                            bank_name_extracted = bank_name_extracted + (" " if bank_name_extracted else "") + extra_for_name
                            bank_name_extracted = bank_name_extracted.strip()
                    def clean_extra_address(address):
                        """
                        Remove extra trailing fragments from the address.
                        """
                        # Remove patterns starting with ",", optional number and a comma, then "Names and Addresses of DCCBs in" till the end
                        cleaned = re.sub(r',\s*(?:\d+\s*,\s*,)?\s*(?:Names and Addresses of DCCBs in|Addresses).*', '', address, flags=re.I)
                        cleaned = re.sub(r',\s*$', '', cleaned)
                        cleaned = re.sub(r'(-\s*\d{3}\s*\d{3}).*', r'\1', cleaned)
                        return cleaned.strip()
                    
                    address_extracted = clean_extra_address(address_extracted)
                    
                    # Remove duplicate spaces and double commas
                    address_extracted = re.sub(r'\s+', ' ', address_extracted).strip()
                    address_extracted = re.sub(r',\s*,+', ', ', address_extracted)
                    address_extracted = address_extracted.strip().strip(',')
                    # Remove any punctuation that starts the address
                    address_extracted = remove_leading_punctuation(address_extracted)

                    # handle two zipcodes by splitting the record into two
                    split_records = split_double_zip_record(bank_name_extracted, address_extracted)
                    if split_records:
                        # Process first split record.
                        rec1, rec2 = split_records
                        bank_name_extracted = rec1['Name']
                        address_extracted = rec1['Address_1']
                        # Build and add first record
                        record = {
                            'Name': bank_name_extracted,
                            'Address_1': address_extracted,
                            'City': "",
                            'Zip': "",
                            'ListProcessDate': processdate,
                            'RegCtry': 'IN',
                            'RegCode': 'RBI',
                            'ListCode': reg.split(' ')[-1],
                            'ListName': list_name,
                            'RegulationType': "Regulated",
                            'Website': ""
                        }
                        add_record(record)
                        # For the second record, we might not have a bank name. You can choose to leave it empty
                        # or set it to the bank name of the previous record
                        record = {
                            'Name': rec2['Name'],  # or use the previous record's bank name if desired.
                            'Address_1': rec2['Address_1'],
                            'City': "",
                            'Zip': "",
                            'ListProcessDate': processdate,
                            'RegCtry': 'IN',
                            'RegCode': 'RBI',
                            'ListCode': reg.split(' ')[-1],
                            'ListName': list_name,
                            'RegulationType': "Regulated",
                            'Website': ""
                        }
                        add_record(record)
                    else:
                        # For Bangalore District and Bangalore Rural District
                        bank_name_extracted, address_extracted = fix_bangalore_case(bank_name_extracted, address_extracted)

                        # Build and add the record.
                        record = {
                            'Name': bank_name_extracted,
                            'Address_1': address_extracted,
                            'City': "",
                            'Zip': "",
                            'ListProcessDate': processdate,
                            'RegCtry': 'IN',
                            'RegCode': 'RBI',
                            'ListCode': reg.split(' ')[-1],
                            'ListName': list_name,
                            'RegulationType': "Regulated",
                            'Website': ""
                        }
                        add_record(record)
    else:
        # Locate target_tbody using two strategies:
        target_tbody = None
        tbodies = soup.find_all('tbody')

        # Use list_name in the header text
        for tbody in tbodies:
            first_row = tbody.find('tr')
            if not first_row:
                continue
            header_cell = first_row.find(['td', 'th'])
            if not header_cell:
                continue
            header_text = header_cell.get_text(strip=True).lower()
            if list_name.lower() in header_text:
                target_tbody = tbody
                print("Located table by list_name:", header_text)
                break

        # 2: look for a PDF link in a header 
        if not target_tbody:
            for tbody in tbodies:
                header_cell = tbody.find('td', class_="tableheader")
                if header_cell and header_cell.find('a', href=re.compile(r'\.pdf$', re.I)):
                    target_tbody = tbody
                    print("Located table by PDF header:", header_cell.get_text(strip=True))
                    rows = target_tbody.find_all('tr')
                    if len(rows) < 2:
                        print("No data rows found in the table")
                    else:
                        print("Using list 2 table logic (PDF header detected)")
                        prev_record_exists = False  # flag to indicate a record was already added
                        for row in rows[1:]:  # skip header row
                            cols = row.find_all('td')
                            if len(cols) < 2:
                                continue
                            # Assume first column is serial and second column is bank name (and possibly a split name/address)
                            serial = cols[0].get_text(strip=True)
                            bank_name_text = cols[1].get_text(strip=True)
                            bank_name_text = re.sub(r'^\s*\d+\.\s*', '', bank_name_text)
                            if bank_name_text.strip() in ["#", "1"]:
                            # Either treat this as a header or empty value,
                            # or try to use a continuation row to update the previous record
                                continue

                            # If the serial is empty and we already have a previous record, this is a continuation row.
                            if serial == "" and prev_record_exists:
                                extra_text = bank_name_text
                                if extra_text:
                                    # Append extra text to the previously added bank's name.
                                    sqldict['Name'][-1] = sqldict['Name'][-1] + " " + extra_text
                                    bank_address = ""
                                continue
                            # Otherwise, treat it as a new record.
                            if bank_name_text:
                                bank_address = ""
                                append_record(bank_name_text, bank_address)
                                prev_record_exists = True
                            
                
        else:
            # identify table type by reading the header row
            header_tr = target_tbody.find('tr')
            table_headers = []
            if header_tr:
                header_cells = header_tr.find_all(['th', 'td'])
                table_headers = [cell.get_text(strip=True).lower() for cell in header_cells]

                # logic 1: process rows using existing approach
                if any(row.find('div') for row in target_tbody.find_all('td', class_='tableheader')):
                    rows = target_tbody.find_all('tr', class_='tablecontent1')
                    if rows:
                        print("Using list 1 table logic")
                        for row in rows:
                            cols = [td for td in row.find_all('td', class_=['tablecontent1', 'tablecontent2'])
                                    if 'text1' not in td.get('class', [])]
                            if not cols:
                                continue

                            for td in cols:
                                for br in td.find_all("br"):
                                    br.replace_with("\n")
                                text = td.get_text("\n", strip=True)
                                lines = text.split("\n")
                                lines = [line.strip() for line in lines if line.strip()]
                                if not lines:
                                    continue

                                if 'chairman' in lines[0].lower() and len(lines) > 1:
                                    bank_name = lines[1].split(",")[0].strip()
                                    address_raw = ", ".join(lines[1].split(",")[1:] + lines[2:]).strip()
                                    # Remove leading commas/spaces, trailing dots, double commas and normalize comma spacing
                                    bank_address = re.sub(r'\s+,', ', ',
                                                        re.sub(r',\s*,', ', ',
                                                        re.sub(r'^[,\s]+', '',
                                                        re.sub(r'\.+$', '', address_raw)))).strip()
                                else:
                                    bank_name = lines[0].split(",")[0].strip()
                                    bank_address = " ".join(lines[1:]).replace(",", "")

                                if bank_name:
                                    sqldict['Name'].append(bank_name)
                                    sqldict['Address_1'].append(bank_address)
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['RegCtry'].append('IN')
                                    sqldict['RegCode'].append('RBI')
                                    sqldict['ListCode'].append(reg.split(' ')[-1])
                                    sqldict['ListName'].append(list_name)
                                    sqldict['RegulationType'].append("Regulated")
                                for key in sqldict:
                                    if len(sqldict[key]) < len(sqldict['ListProcessDate']):
                                        sqldict[key].append('')

                # elif target_tbody.find_all('tr', class_=re.compile(r'tablecontent')):
                #     rows = target_tbody.find_all('tr', class_=re.compile(r'tablecontent'))
                #     if rows:
                #         print("Using logic for processing all data rows regardless of headers")
                #         for row in rows:
                #             cols = row.find_all('td')
                #             # Process left bank details from second column, if available.
                #             if len(cols) >= 2:
                #                 bank_name, bank_address = process_bank_details(cols[1])
                #                 if bank_name:
                #                     append_record(bank_name, bank_address)
                #             # Process right bank details from fourth column, if available.
                #             if len(cols) >= 4:
                #                 bank_name, bank_address = process_bank_details(cols[3])
                #                 if bank_name:
                #                     append_record(bank_name, bank_address)
                    else:
                        print("No data rows found with class 'tablecontent'")

    if list_name.lower() == "regional rural banks":
        print("Processing Regional Rural Banks (List 10)")
        rows = target_tbody.find_all("tr")  # Get all rows without class filter
        for row in rows:
            cells = row.find_all("td")  # Get all cells
            for cell in cells:
                bank_name, full_address, website = extract_rrb_bank_details_list10(cell)
                if bank_name:
                    # Clean website from address
                    full_address = re.sub(r',?\s*(?:www\.[^,\s]+|Website:.*?(?=,|$))', '', full_address, flags=re.I)
                    full_address = re.sub(r',?\s*\d{3,4}-\d{6,8}(?=,|$)', '', full_address)
                    
                    # Clean up formatting
                    full_address = re.sub(r'\s+', ' ', full_address)
                    bank_name = re.sub(r'\s+', ' ', bank_name)
                    full_address = re.sub(r',\s*,', ',', full_address)
                    full_address = full_address.strip(' ,')
                    
                    record = {
                        'Name': bank_name,
                        'Address_1': full_address,
                        'City': "",
                        'Zip': "",
                        'ListProcessDate': processdate,
                        'RegCtry': 'IN',
                        'RegCode': 'RBI',
                        'ListCode': reg.split(' ')[-1],
                        'ListName': list_name,
                        'RegulationType': "Regulated",
                        'Website': website
                    }
                    add_record(record)
    elif target_tbody:
        header_tr = target_tbody.find('tr')
        table_headers = []
        if header_tr:
            header_cells = header_tr.find_all(['th', 'td'])
            table_headers = [cell.get_text(strip=True).lower() for cell in header_cells]
        if any(row.find('div') for row in target_tbody.find_all('td', class_='tableheader')):
            rows = target_tbody.find_all('tr', class_='tablecontent1')
            if rows:
                print("Using list 1 table logic")
                for row in rows:
                    cols = [td for td in row.find_all('td', class_=['tablecontent1','tablecontent2'])
                            if 'text1' not in td.get('class',[])]
                    if not cols:
                        continue
                    for td in cols:
                        for br in td.find_all("br"):
                            br.replace_with("\n")
                        text = td.get_text("\n", strip=True)
                        lines = [line.strip() for line in text.split("\n") if line.strip()]
        elif target_tbody.find_all('tr', class_=re.compile(r'tablecontent')) and reg != "IN RBI 7" and reg != "IN RBI 8" and reg != "IN RBI 9":
            rows = target_tbody.find_all('tr', class_=re.compile(r'tablecontent'))
            if rows:
                print("Using logic for processing all data rows regardless of headers")
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) >= 2:
                        bank_name, bank_address = process_bank_details(cols[1])
                        if bank_name:
                            bank_address = re.sub(r'\s*#\s*', ' ', bank_address)
                            append_record(bank_name, bank_address)
                    if len(cols) >= 4:
                        bank_name, bank_address = process_bank_details(cols[3])
                        if bank_name:
                            bank_address = re.sub(r'\s*#\s*', ' ', bank_address)
                            append_record(bank_name, bank_address)



Working with IN RBI 10
Located table by list_name: regional rural banks
Processing Regional Rural Banks (List 10)


In [6]:

df = pd.DataFrame(sqldict)
df = df[~df['Name'].str.match(r'^\d+\.\s*the\s+chairman$', case=False, na=False)]
df.to_excel(writer, sheet_name='SQL Ready', index=False)
for file in os.listdir(tempfolder):
    file_path = os.path.join(tempfolder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)
writer.close()
driver.quit()